In [ ]:
# Installs the correct modules to a users system using magic

%pip install -q pandas
%pip install -q matplotlib
%pip install -q scikit-learn


# Required Packages
import pandas as pd
import os
import matplotlib.pyplot as plt
import warnings
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn import tree

warnings.filterwarnings("ignore")

In [ ]:
# PREPARE THE DATA: 
# ---------------------------------------------------------

# Gather all json in the directory and concatenate into a dataframe
paths = ['./Data/Luna - Spotify Streaming History', './Data/Kit - Spotify Streaming History', './Data/Madison - Spotify Streaming History', './Data/Complete Streaming History']
folder_path = paths[0]

# Collect data files
json_files = [f for f in os.listdir(folder_path) if f.endswith('.json')]

# Add all .json files to a DataFrame
df_list = []
for file_name in json_files:
    full_path = os.path.join(folder_path, file_name)
    df_list.append(pd.read_json(full_path))

df = pd.concat(df_list, ignore_index=True)

df = df.reset_index()
# Drop unused columns
df = df.drop(['audiobook_uri', 'audiobook_title', 'spotify_track_uri', 'ip_addr', 'platform', 'conn_country', 'index', 'episode_name', 'episode_show_name', 'spotify_episode_uri', 'audiobook_chapter_uri', 'audiobook_chapter_title', 'offline_timestamp', 'incognito_mode', 'offline'], axis=1)


# Convert string to datetime
df['ts'] = pd.to_datetime(df['ts'], utc=True).dt.tz_convert('America/New_York')
#Sort values and reset index
df = df.sort_values(by='ts')

In [ ]:
# Track Listen Counter
def track_counts(df, start_date='1000-01-01', end_date='9999-12-31', ascending=False):
    return df[(df['ts'] >= min(start_date, end_date)) & (df['ts'] < max(start_date, end_date))]['master_metadata_track_name'].value_counts().sort_values(ascending=ascending)

# Album Listen Counter
def album_counts(df, start_date='1000-01-01', end_date='9999-12-31', ascending=False):
    return df[(df['ts'] >= min(start_date, end_date)) & (df['ts'] < max(start_date, end_date))]['master_metadata_album_album_name'].value_counts().sort_values(ascending=ascending)

# Artist Listen Counter
def artist_counts(df, start_date='1000-01-01', end_date='9999-12-31', ascending=False):
    return df[(df['ts'] >= min(start_date, end_date)) & (df['ts'] < max(start_date, end_date))]['master_metadata_album_artist_name'].value_counts().sort_values(ascending=ascending)


In [ ]:
# Printing results of listening counts
def print_listening_counts():
    print('Track Listen Counts:\n', track_counts(df).head(), '\n\n\n')
    print('Artist Listen Counts:\n', artist_counts(df).head(), '\n\n\n')
    print('Album Listen Counts:\n', album_counts(df).head(), '\n\n\n')

# print_listening_counts()

In [ ]:
# Creates a bar graph of a user's top 30 tracks

song_counts = track_counts(df)

song_counts[:30].plot.bar(width=0.6, figsize=(15, 5), rot=87)
plt.title("Top 30 Songs")
plt.xlabel("Song Name")
plt.ylabel("Listening Count")
plt.show()

In [ ]:
# Creates a bar graph of a user's top 30 artists

artist_tally = artist_counts(df)

artist_tally[:30].plot.bar(width=0.6, figsize=(15, 5), rot=87)
plt.title("Top 30 Artists")
plt.xlabel("Artist Name")
plt.ylabel("Listening Count")

In [ ]:
# Creates a histogram of the most frequent listening time in the dataset

time_set = df[(df['ts'] >= '01-01-2000') & (df['ts'] < '01-01-2026')]['ts'].dt.hour

plt.hist(time_set, bins=24, range=(0, 24))
plt.xlabel("Hour of Day")
plt.ylabel("Listening Count")
plt.title("Songs Listened to by Hour of Day")

In [164]:
# Creates DataFrame to calculate skip rate for songs

skipped = df[df['skipped']]['master_metadata_track_name'].value_counts().sort_values(ascending=False)
song_counts = track_counts(df)

track_stats = pd.DataFrame({'total_listens': song_counts, 'skips': skipped})
# track_stats['skips'] = track_stats['skips'].fillna(0)

# Filters tracks with at least 10 listens
track_stats_filtered = track_stats[track_stats['total_listens'] >= 10]

# Calculates skips/total_listens to get skip rates
track_stats_filtered['skip_rate'] = track_stats_filtered['skips'] / track_stats_filtered['total_listens']

# Sort by skip rate
track_stats = track_stats_filtered.sort_values(by='skip_rate', ascending=False)

# Map artists to song
artist_map = df.drop_duplicates(subset=['master_metadata_track_name']).set_index('master_metadata_track_name')['master_metadata_album_artist_name']
track_stats['artist'] = track_stats.index.map(artist_map)

# Set df index
track_stats.index.name = 'track_name'
track_stats_final = track_stats.reset_index()

track_stats_final = track_stats_final[['track_name', 'artist', 'total_listens', 'skips', 'skip_rate']]
track_stats_final[:30]

,track_name,artist,total_listens,skips,skip_rate
0,Água de Beber,Astrud Gilberto,10,10.0,1.000000
1,"Goodnite, Dr. Death",My Chemical Romance,11,10.0,0.909091
2,glacier meadow,Cavetown,10,9.0,0.900000
3,Secret Tunnel,I Kill Giants,16,14.0,0.875000
4,Slowly,The Altogether,13,11.0,0.846154
5,It Starts With Sorry,Andrew Underberg,32,27.0,0.843750
6,UH OH! (feat. BENEE),Sub Urban,20,16.0,0.800000
7,mona lisa,mxmtoon,15,11.0,0.733333
8,Mako,Ax and the Hatchetmen,11,8.0,0.727273
9,Superfan,Ricky Montgomery,11,8.0,0.727273


In [ ]:
# Good idea to only use with one users data, it's pretty hard to predict 3 completely different people

# Used 3 different steps to predict a users skip rate

# 1. ORGANIZE THE DATA
# ---------------------------------------------------------
# Copy over the data to a new df
ml_df = df.dropna(subset=['skipped', 'reason_start', 'master_metadata_track_name']).copy()

# Map the calculated skip rate for each song to the DataFrame
ml_df['historical_skip_rate'] = ml_df['master_metadata_track_name'].map(track_stats['skip_rate']).fillna(0)

# Convert 'reason_start' to dummy variables
ml_df = pd.get_dummies(ml_df, columns=['reason_start'], drop_first=True)

# Define our Features (X) and our Target (y)
# Set our features to reason_start and our target to historical_skip_rate
feature_cols = ['historical_skip_rate'] + [col for col in ml_df.columns if 'reason_start_' in col]
X = ml_df[feature_cols]

# 1 if Skipped, 0 if Not Skipped
y = ml_df['skipped'].astype(int) 


# 2. SPLIT DATA
# ---------------------------------------------------------
# Split the X and y data between training and testing sets. 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# 3. TRAINING
# ---------------------------------------------------------
# Use RandomForestClassifier to create our model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)

# Train the model on your data
rf_model.fit(X_train, y_train)



y_pred = rf_model.predict(X_test)

# Determine the accuracy of our predicted values
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%\n")

# Determine which feature influenced our predictions the most
feature_importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("What drives your skips?")
print(feature_importances)

In [ ]:
# TODO: Update Documentation
clf_0 = rf_model.estimators_[0]

plt.figure(figsize=(80, 15)) 
tree_plot_0 = tree.plot_tree(clf_0, fontsize=15, filled=True)
plt.show()